# 06. Model Evaluation and Translation Inference

This notebook evaluates trained NMT models and performs qualitative & quantitative translation analysis:
1. Loads vocabulary dictionaries and initializes model architecture.
2. Loads saved checkpoints (`seq2seq_sanity_100_batches.pt` or fully trained checkpoints).
3. Computes validation loss with teacher forcing disabled (`ratio = 0`).
4. Performs greedy decoding inference on sample English sentences.
5. Sets up quantitative evaluation with **BLEU** and **chrF** metrics.


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Load Vocabularies & Model Checkpoint

In [ ]:
import torch
import torch.nn as nn
from src.data.vocabulary import load_vocab, PAD_IDX
from src.models.encoder import Encoder
from src.models.decoder import Decoder
from src.models.seq2seq import Seq2Seq
from src.training.checkpoint import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eng_vocab = load_vocab(paths.eng_vocab_path)
amh_vocab = load_vocab(paths.amh_vocab_path)
amh_vocab_inv = {idx: token for token, idx in amh_vocab.items()}

INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(amh_vocab)

encoder = Encoder(INPUT_DIM, 256, 512, 1, 0.2, pad_idx=PAD_IDX)
decoder = Decoder(OUTPUT_DIM, 256, 512, 1, 0.2, pad_idx=PAD_IDX)
model = Seq2Seq(encoder, decoder, device=device).to(device)

checkpoint_path = paths.models_dir / "seq2seq_sanity_100_batches.pt"
if checkpoint_path.exists():
    checkpoint = load_checkpoint(checkpoint_path, model=model, device=device)
    print(f"Loaded checkpoint from: {checkpoint_path}")
    print(f"Checkpoint recorded loss: {checkpoint.get('loss', 'N/A')}")
else:
    print(f"Checkpoint not found at {checkpoint_path}. Using initial weights.")


## 2. Evaluation Loss on Validation Split

In [ ]:
import pandas as pd
from src.data.dataset import TranslationDataset, get_dataloader
from src.training.evaluate import evaluate_epoch

val_df = pd.read_csv(paths.val_filtered_path)
val_dataset = TranslationDataset(val_df, eng_vocab, amh_vocab, max_len=70)
# Use subset for quick evaluation demonstration
val_subset = torch.utils.data.Subset(val_dataset, range(min(500, len(val_dataset))))
val_loader = get_dataloader(val_subset, batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
val_loss = evaluate_epoch(model, val_loader, criterion, device=device)
print(f"Validation Loss (Sample): {val_loss:.4f}")


## 3. Qualitative Translation with Greedy Decoding

In [ ]:
from src.inference.translator import translate_sentence

test_df = pd.read_csv(paths.test_filtered_path)

print("Sample Translations:\n" + "=" * 70)
for i in range(5):
    row = test_df.iloc[i]
    en_sent = row["eng"]
    am_true = row["amh"]

    translated_tokens, _ = translate_sentence(
        sentence=en_sent,
        model=model,
        src_vocab=eng_vocab,
        trg_vocab_inv=amh_vocab_inv,
        max_len=70,
        device=device
    )
    predicted = " ".join(translated_tokens)

    print(f"Source (EN): {en_sent}")
    print(f"Reference:   {am_true}")
    print(f"Predicted:   {predicted}")
    print("-" * 70)


## 4. Quantitative Metrics Framework (BLEU & chrF)
We demonstrate how corpus-level BLEU and chrF scores are calculated across reference and hypothesis translations.

In [ ]:
import sacrebleu

# Example evaluation demonstration
references = [["ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?"]]
hypotheses = ["ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?"]

bleu = sacrebleu.corpus_bleu(hypotheses, references)
chrf = sacrebleu.corpus_chrf(hypotheses, references)

print(f"BLEU score: {bleu.score:.2f}")
print(f"chrF score: {chrf.score:.2f}")


## 5. Next Steps & Planned Architecture Comparison
- **Milestone 1**: Baseline LSTM Seq2Seq model (completed sanity run & established pipeline).
- **Milestone 2**: Full training for baseline model across all 532k pairs.
- **Milestone 3**: Attention-based Seq2Seq architecture (Bahdanau / Luong attention mechanism).
- **Milestone 4**: Comparative evaluation matrix (BLEU, chrF, Latency, Attention Heatmaps).
